# Day 9 — QA Pipelines with HuggingFace

**Roadmap Phase:** Phase 2 — Applied NLP  
**Objective:** Build a working Question-Answering pipeline using extractive QA models  
**Commit target:** `feat: day09 - QA pipelines with HuggingFace`

---

### What you'll build
- Load a pretrained extractive QA model (`deepset/roberta-base-squad2`)
- Run 5+ context/question pairs through the pipeline
- Log results in a clean table: question | answer | score
- Probe edge cases: unanswerable questions, ambiguous context

### 15-minute rule
If anything is blocking you for >15 min, write it down and move on. Log it at the bottom.

## 0. Setup & Imports

In [ ]:
# Install if needed (run once)
# !pip install transformers torch pandas

In [ ]:
from transformers import pipeline
import pandas as pd

print("Imports OK")

## 1. Load the QA Pipeline

We use `deepset/roberta-base-squad2` — a RoBERTa model fine-tuned on SQuAD2.0.  
SQuAD2 includes **unanswerable questions**, which makes this model better at saying "I don't know".

In [ ]:
qa_pipeline = pipeline(
    task="question-answering",
    model="deepset/roberta-base-squad2"
)

print("Model loaded:", qa_pipeline.model.config.model_type)

## 2. Define Contexts

Use a variety of domains so you see how the model generalises.

In [ ]:
contexts = {
    "nepal": """
        Nepal is a landlocked country in South Asia, bordered by China to the north
        and India to the south, east, and west. Kathmandu is the capital and largest city.
        Nepal is home to eight of the world's ten tallest mountains, including Mount Everest,
        the highest point on Earth at 8,848 metres. The country has a population of
        approximately 30 million people and covers an area of 147,181 square kilometres.
    """,

    "transformers": """
        Transformer models were introduced in the paper 'Attention Is All You Need' by
        Vaswani et al. in 2017. They rely entirely on self-attention mechanisms and
        eliminate recurrence and convolutions. BERT, released by Google in 2018,
        was one of the first large-scale pretrained transformers for NLP tasks.
        GPT-2 was released by OpenAI in 2019 and demonstrated impressive text generation.
    """,

    "rag": """
        Retrieval-Augmented Generation (RAG) combines a retrieval system with a
        generative language model. The retrieval step fetches relevant documents
        from a knowledge base using dense or sparse vector search. These documents
        are then passed as context to the generator, which produces the final answer.
        FAISS is a popular library for fast approximate nearest-neighbour search
        used in the retrieval step.
    """,

    "python": """
        Python was created by Guido van Rossum and first released in 1991.
        It emphasises code readability and simplicity. Python 3 was released
        in 2008 and is the current major version. Python is widely used in
        data science, machine learning, web development, and automation.
        The Python Package Index (PyPI) hosts over 400,000 packages.
    """,
}

print("Contexts defined:", list(contexts.keys()))

## 3. Define Questions

Mix of: answerable, tricky, and **unanswerable** (to probe the model's confidence).

In [ ]:
# Each entry: (context_key, question, expected_type)
# expected_type: 'answerable' | 'unanswerable' | 'ambiguous'

qa_pairs = [
    # Straightforward
    ("nepal",        "What is the capital of Nepal?",                        "answerable"),
    ("nepal",        "How tall is Mount Everest?",                           "answerable"),
    ("transformers", "Who introduced the Transformer architecture?",         "answerable"),
    ("transformers", "When was BERT released?",                              "answerable"),
    ("rag",          "What library is used for nearest-neighbour search?",   "answerable"),
    ("python",       "Who created Python?",                                  "answerable"),

    # Edge cases — unanswerable / information not in context
    ("nepal",        "What is the currency of Nepal?",                       "unanswerable"),
    ("transformers", "What is the parameter count of GPT-2?",                "unanswerable"),
    ("rag",          "What year was RAG first proposed?",                    "unanswerable"),
    ("python",       "What is the latest stable version of Python?",         "unanswerable"),
]

print(f"{len(qa_pairs)} QA pairs ready")

## 4. Run the Pipeline & Collect Results

In [ ]:
results = []

for ctx_key, question, expected_type in qa_pairs:
    context = contexts[ctx_key].strip()

    output = qa_pipeline(
        question=question,
        context=context,
        handle_impossible_answer=True   # SQuAD2 feature: returns empty string if unanswerable
    )

    results.append({
        "context":       ctx_key,
        "question":      question,
        "answer":        output["answer"] if output["answer"] else "[no answer]",
        "score":         round(output["score"], 4),
        "expected_type": expected_type,
    })

df = pd.DataFrame(results)
print(f"Done. {len(df)} results collected.")

## 5. Results Table

In [ ]:
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", "{:.4f}".format)

df[["context", "question", "answer", "score", "expected_type"]]

## 6. Analysis

Answer the following before moving on.

In [ ]:
# Score stats by expected type
print("=== Mean confidence score by type ===")
print(df.groupby("expected_type")["score"].agg(["mean", "min", "max"]).round(4))

print("\n=== Low-confidence answers (score < 0.3) ===")
low_conf = df[df["score"] < 0.3][["question", "answer", "score"]]
print(low_conf if not low_conf.empty else "None")

## 7. Single Query Helper

Interactive cell — change the question and context and re-run.

In [ ]:
# ── Edit these two strings and re-run ─────────────────────────────────────
MY_CONTEXT = """
    Sentence transformers are models that map sentences to dense vector embeddings.
    They are trained using siamese or triplet networks on NLI or semantic similarity datasets.
    The all-MiniLM-L6-v2 model is a popular lightweight choice for semantic search tasks.
"""

MY_QUESTION = "What kind of network is used to train sentence transformers?"
# ──────────────────────────────────────────────────────────────────────────

out = qa_pipeline(question=MY_QUESTION, context=MY_CONTEXT.strip(), handle_impossible_answer=True)
print(f"Q: {MY_QUESTION}")
print(f"A: {out['answer'] or '[no answer]'}")
print(f"Score: {out['score']:.4f}")

## 8. Daily Log

Fill this in before committing.

```
Roadmap phase   : Phase 2 — Applied NLP
Objective       : QA pipeline with extractive QA model
What was built  : 
Struggles       : 
Next task       : Day 10 — Summarization
Energy level    : /5
15-min rule     : triggered? Y/N — what was the blocker?
One blocker     : 
```

---

## Key Concepts Recap

| Concept | What it means |
|---------|---------------|
| Extractive QA | Model selects a span from the context — doesn't generate new text |
| `score` field | Model's confidence that the answer span is correct |
| SQuAD2 | Dataset with unanswerable questions — trains model to abstain |
| `handle_impossible_answer=True` | Returns empty string + low score if no answer found in context |
| Next step | Day 10: Summarization — abstractive generation, not span extraction |